# 02 Exploratory Data Analysis

This notebook explores the datasets prepared in `01_load_data.ipynb`.

The goal is to understand data coverage, regional availability, missing values, indicator definitions, and candidate variables before building the AI Literacy Gap Index.  
At this stage, I separate exploratory analysis from final modelling decisions. The final index variables should only be selected after inspecting the available data.

- Step 1: Load each dataset separately
- Step 2: Inspect each dataset separately
- Step 3: Select candidate variables from each dataset
- Step 4: Filter each dataset to the chosen variable / year / geo level
- Step 5: Convert each dataset into one clean NUTS-1 feature table
- Step 6: Merge those feature tables into one final modelling dataframe

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

# -----------------------------
# Project paths
# -----------------------------
# This notebook is located in: CHL_DATA_SCIENCE_PROJECT/notebooks/
# The project root is therefore one level above the notebook folder.

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_DIR / "results"
EDA_DIR = RESULTS_DIR / "eda"

for path in [RAW_DIR, PROCESSED_DIR, EDA_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Processed data directory:", PROCESSED_DIR)
print("EDA results directory:", EDA_DIR)

# -----------------------------
# Load project data catalog
# -----------------------------
catalog_path = PROCESSED_DIR / "project_data_catalog.csv"

if not catalog_path.exists():
    raise FileNotFoundError(
        f"Data catalog not found at {catalog_path}. "
        "Please run 01_load_data.ipynb first."
    )

data_catalog = pd.read_csv(catalog_path)

display(data_catalog)

Project directory: c:\Users\Lu\OneDrive\ToU\chl_data_science_project
Processed data directory: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed
EDA results directory: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\results\eda


,dataset_code,dataset_name,pillar,processed_file,main_use,file_exists,processed_path,n_rows,n_columns,min_year,max_year,n_years,main_value_column,missing_share_main_value,n_geo_codes,n_nuts1_codes,n_nuts2_codes,n_country_like_codes,n_other_geo_codes,columns
0,isoc_r_dskl_i,Digital skills by NUTS-1 region,Digital readiness,digital_skills_nuts1_labeled_filtered.csv,Core proxy for digital readiness / AI literacy...,True,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,4126,13,2025,2025,1,value,0.005817,88,88,0,0,0,"freq, indic_is, unit, geo, year, value_raw, va..."
1,tgs00107,People at risk of poverty or social exclusion,Social vulnerability,poverty_social_exclusion_nuts1_weighted.csv,Population-weighted NUTS-1 vulnerability indic...,True,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,695,10,2015,2025,11,poverty_social_exclusion_rate,0.000000,91,91,0,0,0,"geo, nuts1_name, country_code, year, poverty_s..."
2,demo_r_d2jan,"Population by age, sex and region",Population weights / demographics,population_nuts2_total_long.csv,Population weights for NUTS-2 to NUTS-1 aggreg...,True,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,10620,12,1990,2025,36,population,0.155179,295,0,295,0,0,"freq, unit, sex, age, geo, year, population_ra..."
3,edat_lfse_04,Educational attainment by region,Education,education_attainment_labeled.csv,Education structure as AI literacy readiness f...,True,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,939900,15,2000,2025,26,value,0.317754,511,111,289,36,111,"freq, sex, isced11, age, unit, geo, year, valu..."
4,trng_lfse_04,Participation in education and training,Adult learning,lifelong_learning_labeled.csv,Lifelong learning / reskilling capacity,True,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,79716,13,2000,2025,26,value,0.144237,511,111,289,36,111,"freq, unit, sex, age, geo, year, value_raw, va..."
5,lfst_r_lfu3pers,Unemployment by education level and region,Labour market vulnerability,unemployment_education_region_labeled.csv,Labour market risk by education level,True,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,1870155,15,1999,2025,27,value,0.485299,512,111,290,36,111,"freq, isced11, sex, age, unit, geo, year, valu..."
6,isoc_r_eb_ain2,Enterprise AI adoption by region,AI exposure,enterprise_ai_adoption_labeled.csv,Regional AI adoption pressure from enterprises,True,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,176730,15,2023,2025,3,value,0.344548,179,38,106,34,35,"freq, nace_r2, size_emp, indic_is, unit, geo, ..."


## Load prepared datasets

The data loading notebook saved each dataset as a separate processed CSV file.  
In this step, I load all available processed datasets into a dictionary and create a compact overview of their shapes, time coverage, regional coverage, and missing values.

This gives a first check of which datasets are ready for EDA and which may need additional filtering or cleaning.

In [5]:
# -----------------------------
# Load all prepared datasets from the project catalog
# -----------------------------

datasets = {}
dataset_overview_rows = []

for _, row in data_catalog.iterrows():
    dataset_code = row["dataset_code"]
    processed_file = row["processed_file"]
    file_path = PROCESSED_DIR / processed_file

    if not file_path.exists():
        print(f"Skipping {dataset_code}: file not found at {file_path}")
        continue

    df = pd.read_csv(file_path)
    datasets[dataset_code] = df

    # Identify possible main value columns
    possible_value_cols = [
        "value",
        "population",
        "poverty_social_exclusion_rate"
    ]
    available_value_cols = [col for col in possible_value_cols if col in df.columns]
    main_value_col = available_value_cols[0] if available_value_cols else None

    overview = {
        "dataset_code": dataset_code,
        "dataset_name": row["dataset_name"],
        "pillar": row["pillar"],
        "n_rows": df.shape[0],
        "n_columns": df.shape[1],
        "main_value_col": main_value_col,
        "missing_share_main_value": df[main_value_col].isna().mean() if main_value_col else np.nan,
        "n_geo_codes": df["geo"].nunique() if "geo" in df.columns else np.nan,
        "min_year": df["year"].min() if "year" in df.columns else np.nan,
        "max_year": df["year"].max() if "year" in df.columns else np.nan,
        "n_years": df["year"].nunique() if "year" in df.columns else np.nan,
    }

    dataset_overview_rows.append(overview)

dataset_overview = pd.DataFrame(dataset_overview_rows)

print("Loaded datasets:")
display(list(datasets.keys()))

display(dataset_overview)

Loaded datasets:


['isoc_r_dskl_i',
 'tgs00107',
 'demo_r_d2jan',
 'edat_lfse_04',
 'trng_lfse_04',
 'lfst_r_lfu3pers',
 'isoc_r_eb_ain2']

,dataset_code,dataset_name,pillar,n_rows,n_columns,main_value_col,missing_share_main_value,n_geo_codes,min_year,max_year,n_years
0,isoc_r_dskl_i,Digital skills by NUTS-1 region,Digital readiness,4126,13,value,0.005817,88,2025,2025,1
1,tgs00107,People at risk of poverty or social exclusion,Social vulnerability,695,10,poverty_social_exclusion_rate,0.000000,91,2015,2025,11
2,demo_r_d2jan,"Population by age, sex and region",Population weights / demographics,10620,12,population,0.155179,295,1990,2025,36
3,edat_lfse_04,Educational attainment by region,Education,939900,15,value,0.317754,511,2000,2025,26
4,trng_lfse_04,Participation in education and training,Adult learning,79716,13,value,0.144237,511,2000,2025,26
5,lfst_r_lfu3pers,Unemployment by education level and region,Labour market vulnerability,1870155,15,value,0.485299,512,1999,2025,27
6,isoc_r_eb_ain2,Enterprise AI adoption by region,AI exposure,176730,15,value,0.344548,179,2023,2025,3


## Load processed datasets into separate dataframes

Each prepared dataset is loaded into its own dataframe.  
At this stage, I keep the datasets separate because they differ in regional level, time coverage, indicators, units, and structure. A combined feature table will only be created later after the relevant variables have been inspected and selected.

In [6]:
# -----------------------------
# Load each processed dataset into its own dataframe
# -----------------------------

digital_skills_df = pd.read_csv(PROCESSED_DIR / "digital_skills_nuts1_labeled_filtered.csv")
poverty_df = pd.read_csv(PROCESSED_DIR / "poverty_social_exclusion_nuts1_weighted.csv")
population_df = pd.read_csv(PROCESSED_DIR / "population_nuts2_total_long.csv")
education_df = pd.read_csv(PROCESSED_DIR / "education_attainment_labeled.csv")
lifelong_learning_df = pd.read_csv(PROCESSED_DIR / "lifelong_learning_labeled.csv")
unemployment_df = pd.read_csv(PROCESSED_DIR / "unemployment_education_region_labeled.csv")
enterprise_ai_df = pd.read_csv(PROCESSED_DIR / "enterprise_ai_adoption_labeled.csv")

# Store them in a dictionary as well for easier iteration
dataframes = {
    "digital_skills_df": digital_skills_df,
    "poverty_df": poverty_df,
    "population_df": population_df,
    "education_df": education_df,
    "lifelong_learning_df": lifelong_learning_df,
    "unemployment_df": unemployment_df,
    "enterprise_ai_df": enterprise_ai_df,
}

# -----------------------------
# Inspect structure of each dataframe
# -----------------------------

for name, df in dataframes.items():
    print("=" * 100)
    print(name)
    print("=" * 100)
    
    print("\nDataFrame info:")
    df.info()
    
    print("\nFirst rows:")
    display(df.head())
    
    print("\n")

digital_skills_df

DataFrame info:
<class 'pandas.DataFrame'>
RangeIndex: 4126 entries, 0 to 4125
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   freq            4126 non-null   str    
 1   indic_is        4126 non-null   str    
 2   unit            4126 non-null   str    
 3   geo             4126 non-null   str    
 4   year            4126 non-null   int64  
 5   value_raw       4126 non-null   str    
 6   value           4102 non-null   float64
 7   indic_is_label  4126 non-null   str    
 8   unit_label      4126 non-null   str    
 9   geo_label       4126 non-null   str    
 10  nuts1_name      4126 non-null   str    
 11  country_code    4126 non-null   str    
 12  nuts_level      4126 non-null   int64  
dtypes: float64(1), int64(2), str(10)
memory usage: 419.2 KB

First rows:


,freq,indic_is,unit,geo,year,value_raw,value,indic_is_label,unit_label,geo_label,nuts1_name,country_code,nuts_level
0,A,I_DSK2_AB,PC_IND,AT1,2025,36.81,36.81,Individuals with above basic overall digital s...,Percentage of individuals,Ostösterreich,Ostösterreich,AT,1
1,A,I_DSK2_AB,PC_IND,AT2,2025,31.11,31.11,Individuals with above basic overall digital s...,Percentage of individuals,Südösterreich,Südösterreich,AT,1
2,A,I_DSK2_AB,PC_IND,AT3,2025,32.85,32.85,Individuals with above basic overall digital s...,Percentage of individuals,Westösterreich,Westösterreich,AT,1
3,A,I_DSK2_AB,PC_IND,BE1,2025,35.82,35.82,Individuals with above basic overall digital s...,Percentage of individuals,Région de Bruxelles-Capitale/Brussels Hoofdste...,Région de Bruxelles-Capitale/Brussels Hoofdste...,BE,1
4,A,I_DSK2_AB,PC_IND,BE2,2025,29.59,29.59,Individuals with above basic overall digital s...,Percentage of individuals,Vlaams Gewest,Vlaams Gewest,BE,1




poverty_df

DataFrame info:
<class 'pandas.DataFrame'>
RangeIndex: 695 entries, 0 to 694
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   geo                            695 non-null    str    
 1   nuts1_name                     695 non-null    str    
 2   country_code                   695 non-null    str    
 3   year                           695 non-null    int64  
 4   poverty_social_exclusion_rate  695 non-null    float64
 5   population_covered             695 non-null    float64
 6   nuts1_population_total         695 non-null    float64
 7   population_coverage_share      695 non-null    float64
 8   n_nuts2_with_poverty_data      695 non-null    int64  
 9   n_nuts2_population_regions     695 non-null    int64  
dtypes: float64(4), int64(3), str(3)
memory usage: 54.4 KB

First rows:


,geo,nuts1_name,country_code,year,poverty_social_exclusion_rate,population_covered,nuts1_population_total,population_coverage_share,n_nuts2_with_poverty_data,n_nuts2_population_regions
0,AL0,Shqipëria,AL,2017,58.497780,2876591.0,2876591.0,1.0,3,3
1,AL0,Shqipëria,AL,2018,53.959695,2870324.0,2870324.0,1.0,3,3
2,AL0,Shqipëria,AL,2019,50.717062,2862427.0,2862427.0,1.0,3,3
3,AL0,Shqipëria,AL,2020,46.186028,2845955.0,2845955.0,1.0,3,3
4,AL0,Shqipëria,AL,2021,46.598042,2829741.0,2829741.0,1.0,3,3




population_df

DataFrame info:
<class 'pandas.DataFrame'>
RangeIndex: 10620 entries, 0 to 10619
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   freq               10620 non-null  str    
 1   unit               10620 non-null  str    
 2   sex                10620 non-null  str    
 3   age                10620 non-null  str    
 4   geo                10620 non-null  str    
 5   year               10620 non-null  int64  
 6   population_raw     10620 non-null  str    
 7   population         8972 non-null   float64
 8   nuts2_name         10620 non-null  str    
 9   parent_nuts1       10620 non-null  str    
 10  parent_nuts1_name  10620 non-null  str    
 11  country_code       10620 non-null  str    
dtypes: float64(1), int64(1), str(10)
memory usage: 995.8 KB

First rows:


...

,freq,unit,sex,age,geo,year,value_raw,value,freq_label,unit_label,sex_label,age_label,geo_label
0,A,PC,F,Y18-64,AT,2000,12.2,12.2,Annual,Percentage,Females,From 18 to 64 years,Austria
1,A,PC,F,Y18-64,AT1,2000,12.0,12.0,Annual,Percentage,Females,From 18 to 64 years,Ostösterreich
2,A,PC,F,Y18-64,AT11,2000,8.4,8.4,Annual,Percentage,Females,From 18 to 64 years,Burgenland
3,A,PC,F,Y18-64,AT12,2000,10.9,10.9,Annual,Percentage,Females,From 18 to 64 years,Niederösterreich
4,A,PC,F,Y18-64,AT13,2000,13.5,13.5,Annual,Percentage,Females,From 18 to 64 years,Wien




unemployment_df

DataFrame info:
<class 'pandas.DataFrame'>
RangeIndex: 1870155 entries, 0 to 1870154
Data columns (total 15 columns):
 #   Column         Dtype  
---  ------         -----  
 0   freq           str    
 1   isced11        str    
 2   sex            str    
 3   age            str    
 4   unit           str    
 5   geo            str    
 6   year           int64  
 7   value_raw      str    
 8   value          float64
 9   freq_label     str    
 10  isced11_label  str    
 11  sex_label      str    
 12  age_label      str    
 13  unit_label     str    
 14  geo_label      str    
dtypes: float64(1), int64(1), str(13)
memory usage: 214.0 MB

First rows:


,freq,isced11,sex,age,unit,geo,year,value_raw,value,freq_label,isced11_label,sex_label,age_label,unit_label,geo_label
0,A,ED0-2,F,Y15-24,THS_PER,AT,1999,6.7 u,6.7,Annual,"Less than primary, primary and lower secondary...",Females,From 15 to 24 years,Thousand persons,Austria
1,A,ED0-2,F,Y15-24,THS_PER,AT1,1999,: u,NaN,Annual,"Less than primary, primary and lower secondary...",Females,From 15 to 24 years,Thousand persons,Ostösterreich
2,A,ED0-2,F,Y15-24,THS_PER,AT11,1999,: u,NaN,Annual,"Less than primary, primary and lower secondary...",Females,From 15 to 24 years,Thousand persons,Burgenland
3,A,ED0-2,F,Y15-24,THS_PER,AT12,1999,: u,NaN,Annual,"Less than primary, primary and lower secondary...",Females,From 15 to 24 years,Thousand persons,Niederösterreich
4,A,ED0-2,F,Y15-24,THS_PER,AT13,1999,: u,NaN,Annual,"Less than primary, primary and lower secondary...",Females,From 15 to 24 years,Thousand persons,Wien




enterprise_ai_df

DataFrame info:
<class 'pandas.DataFrame'>
RangeIndex: 176730 entries, 0 to 176729
Data columns (total 15 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   freq            176730 non-null  str    
 1   nace_r2         176730 non-null  str    
 2   size_emp        176730 non-null  str    
 3   indic_is        176730 non-null  str    
 4   unit            176730 non-null  str    
 5   geo             176730 non-null  str    
 6   year            176730 non-null  int64  
 7   value_raw       176730 non-null  str    
 8   value           115838 non-null  float64
 9   freq_label      176730 non-null  str    
 10  nace_r2_label   176730 non-null  str    
 11  size_emp_label  176730 non-null  str    
 12  indic_is_label  176730 non-null  str    
 13  unit_label      176730 non-null  str    
 14  geo_label       176730 non-null  str    
dtypes: float64(1), int64(1), str(13)
memory usage: 20.2 MB

First rows:


,freq,nace_r2,size_emp,indic_is,unit,geo,year,value_raw,value,freq_label,nace_r2_label,size_emp_label,indic_is_label,unit_label,geo_label
0,A,C,GE10,E_AI_CC,PC_ENT,AT,2023,9.95,9.95,Annual,Manufacturing,10 persons employed or more,Enterprises use AI technologies and buy any cl...,Percentage of enterprises,Austria
1,A,C,GE10,E_AI_CC,PC_ENT,AT1,2023,9.35,9.35,Annual,Manufacturing,10 persons employed or more,Enterprises use AI technologies and buy any cl...,Percentage of enterprises,Ostösterreich
2,A,C,GE10,E_AI_CC,PC_ENT,AT11,2023,5.81,5.81,Annual,Manufacturing,10 persons employed or more,Enterprises use AI technologies and buy any cl...,Percentage of enterprises,Burgenland
3,A,C,GE10,E_AI_CC,PC_ENT,AT12,2023,6.38,6.38,Annual,Manufacturing,10 persons employed or more,Enterprises use AI technologies and buy any cl...,Percentage of enterprises,Niederösterreich
4,A,C,GE10,E_AI_CC,PC_ENT,AT13,2023,17.93,17.93,Annual,Manufacturing,10 persons employed or more,Enterprises use AI technologies and buy any cl...,Percentage of enterprises,Wien
